# Random Forest

For this project we will be exploring publicly available data from [LendingClub.com](www.lendingclub.com). Lending Club connects people who need money (borrowers) with people who have money (investors). Hopefully, as an investor you would want to invest in people who showed a profile of having a high probability of paying you back. We will try to create a model that will help predict this.

Lending club had a [very interesting year in 2016](https://en.wikipedia.org/wiki/Lending_Club#2016), so let's check out some of their data and keep the context in mind. This data is from before they even went public.

We will use lending data from 2007-2010 and be trying to classify and predict whether or not the borrower paid back their loan in full. You can download the data from [here](https://www.lendingclub.com/info/download-data.action) or just use the csv already provided. It's recommended you use the csv provided as it has been cleaned of NA values.

Here are what the columns represent:
* credit.policy: 1 if the customer meets the credit underwriting criteria of LendingClub.com, and 0 otherwise.
* purpose: The purpose of the loan (takes values "credit_card", "debt_consolidation", "educational", "major_purchase", "small_business", and "all_other").
* int.rate: The interest rate of the loan, as a proportion (a rate of 11% would be stored as 0.11). Borrowers judged by LendingClub.com to be more risky are assigned higher interest rates.
* installment: The monthly installments owed by the borrower if the loan is funded.
* log.annual.inc: The natural log of the self-reported annual income of the borrower.
* dti: The debt-to-income ratio of the borrower (amount of debt divided by annual income).
* fico: The FICO credit score of the borrower.
* days.with.cr.line: The number of days the borrower has had a credit line.
* revol.bal: The borrower's revolving balance (amount unpaid at the end of the credit card billing cycle).
* revol.util: The borrower's revolving line utilization rate (the amount of the credit line used relative to total credit available).
* inq.last.6mths: The borrower's number of inquiries by creditors in the last 6 months.
* delinq.2yrs: The number of times the borrower had been 30+ days past due on a payment in the past 2 years.
* pub.rec: The borrower's number of derogatory public records (bankruptcy filings, tax liens, or judgments).
* not.fully.paid: 1 if the loan is not paid in full, and 0 otherwise.

## Learning Objectives

At the end of this notebook, you should be able to:

- Explore the Lending Club data with histograms, a countplot, and faceted regression plots, and interpret what each one shows.
- Encode the categorical `purpose` feature with `pd.get_dummies` and split the data into training and test sets.
- Train and compare a decision tree and a random forest to predict loan defaults.
- Evaluate both models with a classification report and confusion matrix, counter class imbalance with `class_weight`, and rank feature importances.

## Import Libraries

**Import the usual libraries for pandas and plotting. You can import sklearn later on.**

In [1]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
%matplotlib inline
plt.rcParams["patch.force_edgecolor"] = True

## Get the Data

**Use pandas to read loan_data.csv as a dataframe called loans.**

In [2]:
# Your code here!

**Check out the info(), head(), and describe() methods on loans.**

In [1]:
# Your code here!

In [2]:
# Your code here!

In [3]:
# Your code here!

## Exploratory Data Analysis

Let's do some data visualization! We'll use seaborn and pandas built-in plotting capabilities, but feel free to use whatever library you want. Don't worry about the colors matching, just worry about getting the main idea of the plot.

**Create a histogram of two FICO distributions on top of each other, one for each credit.policy outcome.**

*Note: This is pretty tricky, feel free to reference the solutions. You'll probably need one line of code for each histogram, I also recommend just using pandas built in .hist()*

In [4]:
# Your code here!

**Create a similar figure, except this time select by the not.fully.paid column.**

In [5]:
# Your code here!

**Create a countplot using seaborn showing the counts of loans by purpose, with the color hue defined by not.fully.paid.**

In [6]:
# Your code here!

**Before moving on, read this plot. Which loan purposes carry the most not-fully-paid loans in absolute terms, and does any purpose look disproportionately risky relative to its total volume (for example `small_business`)? Write a sentence: a chart is only worth making if you say what it tells you.**

<details>
<summary>Show answer</summary>

`debt_consolidation` and `credit_card` loans dominate the raw counts, so they also contribute the largest absolute number of not-fully-paid loans. But volume is not the same as risk. `small_business` loans carry a much higher *share* of not-fully-paid loans relative to their own total, so they look disproportionately risky even though there are fewer of them. Reading the plot by proportion and not just bar height is what turns a chart into an insight.
</details>

**Let's see the trend between FICO score and interest rate. Recreate the following jointplot.**

In [7]:
# Your code here!

**The `jointplot` above showed the overall FICO vs interest-rate trend. Now let's ask a sharper question: does that trend differ depending on *why* people borrow?**

**Use `sns.lmplot()` to plot `int.rate` against `fico`, faceting into one column per loan `purpose` (use `col='purpose'` with `col_wrap=3`). Pass `scatter_kws={'alpha': 0.3}` so overlapping points stay readable.**

This is more informative than splitting on `credit.policy`, which is itself derived from FICO, so a FICO split there mostly just re-draws the same line. Loan purpose is independent of the score, so the per-purpose regression lines can genuinely differ.

> Reminder: `lmplot()` is a scatter plot with an overlaid regression line (`regplot()`), wrapped in a `FacetGrid()` so you can break the relationship out across subsets. See the [lmplot() docs](https://seaborn.pydata.org/generated/seaborn.lmplot.html) and the [regplot vs lmplot explainer](https://seaborn.pydata.org/tutorial/regression.html).

In [8]:
# Your code here!

**Read your faceted plot. Does the slope of interest rate against FICO look the same for every loan purpose, or are some purposes priced more steeply against credit score than others? Write one or two sentences on what stands out, and on whether the relationship is really as clean as the single `jointplot` made it look.**

<details>
<summary>Show answer</summary>

In every facet the trend is negative: higher FICO scores get lower interest rates, as expected. The slope is not identical across purposes though. `small_business` and `debt_consolidation` are priced more steeply against credit score, while thinner categories like `educational` are flatter and noisier (their regression band is wide because they have far fewer loans). So the relationship is less clean than the single `jointplot` implied: pricing depends partly on *why* someone is borrowing, not on FICO alone. Splitting on `credit.policy` would not reveal this, because `credit.policy` is itself a FICO-based rule.
</details>

## Setting up the Data

Let's get ready to set up our data for our Random Forest Classification Model!

**Check loans.info() again.**

In [9]:
# Your code here!

### Categorical Features

Notice that the **purpose** column as categorical

That means we need to transform them using dummy variables so sklearn will be able to understand them. Let's do this in one clean step using pd.get_dummies.

Let's show you a way of dealing with these columns that can be expanded to multiple categorical features if necessary.

**Create a list of 1 element containing the string 'purpose'. Call this list cat_feats.**

In [10]:
# Your code here!

**Now use pd.get_dummies(loans,columns=cat_feats,drop_first=True) to create a fixed larger dataframe that has new feature columns with dummy variables. Set this dataframe as final_data.**

In [12]:
# Your code here!

### Train Test Split

Now its time to split our data into a training set and a testing set!

**Use sklearn to split your data into a training set and a testing set as we've done in the past.**

In [14]:
from sklearn.model_selection import train_test_split

In [15]:
# Your code here!

## Training a Decision Tree Model

Let's start by training a single decision tree first!

**Import DecisionTreeClassifier**

In [18]:
# Your code here!

**Create an instance of DecisionTreeClassifier() called dtree and fit it to the training data.**

In [19]:
# Your code here!

### Predictions and Evaluation of Decision Tree
**Create predictions from the test set and create a classification report and a confusion matrix.**

In [22]:
from sklearn.metrics import classification_report, confusion_matrix

In [16]:
# Your code here!

## Training the Random Forest model

Now its time to train our model!

**Create an instance of the RandomForestClassifier class and fit it to our training data from the previous step.**

In [25]:
from sklearn.ensemble import RandomForestClassifier

In [18]:
# Your code here!

### Predictions and Evaluation

Let's predict off the y_test values and evaluate our model.

**Predict the class of not.fully.paid for the X_test data.**

In [19]:
# Your code here!

**Now create a classification report from the results. Do you get anything strange or some sort of warning?**

In [20]:
# Your code here!

**Show the Confusion Matrix for the predictions.**

In [21]:
# Your code here!

## Interpreting your Results

**Compare the two classification reports above. The random forest has the higher overall accuracy, but what happened to its recall for class 1 (the loans that were *not* fully paid)? Write a sentence or two: which model is actually more useful to an investor trying to avoid bad loans, and why is accuracy a misleading score here?**

<details>
<summary>Show answer</summary>

The random forest's overall accuracy (about 0.84) beats the decision tree's (about 0.73), but its recall for class 1 collapses to roughly 0.02: it predicts "fully paid" for almost everyone and barely catches a single risky loan. The single tree caught about a quarter of them (recall around 0.24). Because only 16% of loans default, always guessing "fully paid" already scores about 84% accuracy, so accuracy rewards exactly the wrong behaviour here. An investor cares about catching the bad loans, which is recall on class 1, so the high-accuracy forest is of little practical use as trained.
</details>

### Dealing With Class Imbalance

Only about 16% of the loans in this dataset were *not* fully paid. A model can score high accuracy just by predicting "fully paid" almost every time, which is exactly why the random forest above barely flags any of the risky loans (look at its recall for class 1). High accuracy hid a model that is of little practical use for the one thing we care about.

**Retrain the random forest with `class_weight='balanced'` (keep `n_estimators=200` and set `random_state=101` so the result is reproducible), predict on `X_test`, and print the classification report and confusion matrix again.**

In [ ]:
# Your code here!

**Compare this report to the unbalanced one. What happened to the recall for class 1, and what did it cost you elsewhere (precision, overall accuracy)? Every choice here is a trade-off: describe it in your own words, and say which version you would recommend to an investor.**

<details>
<summary>Show answer</summary>

Recall for class 1 rises from about 0.02 to roughly 0.12, so the balanced model finally flags some of the true defaulters (around 55 instead of 9) rather than ignoring them. The cost is real: precision on class 1 stays low (about 0.29) and overall accuracy slips a little (about 0.84 to 0.82), because we now raise more false alarms on loans that were actually repaid. Which version you choose depends on the cost of a missed bad loan versus wrongly rejecting a good borrower. `class_weight='balanced'` is a modest adjustment, not a complete fix: bigger gains would need resampling, threshold tuning, or better features.
</details>

### Which Features Drive the Prediction?

A random forest can tell us how much each feature contributed to its decisions through the `.feature_importances_` attribute. This is one of the things that makes tree ensembles so useful: we can see which inputs drove its predictions.

**Build a pandas Series from your fitted random forest's `feature_importances_`, indexed by `X.columns`, sort it, and plot it as a horizontal bar chart.**

In [ ]:
# Your code here!

**Which three features mattered most? Do they line up with what your EDA plots suggested earlier (for example the FICO and interest-rate relationships)? Write a couple of sentences connecting the model back to the data you explored.**

<details>
<summary>Show answer</summary>

The top features are the continuous numeric columns (`installment`, `revol.util`, `days.with.cr.line`, `revol.bal`, `dti`), while the one-hot `purpose_*` dummies and rare flags sit at the bottom. Surprisingly, `fico` and `int.rate`, which looked most discriminative in the EDA, land only in the middle of the pack. That mismatch is the lesson: default (Gini) importance is biased toward features with many distinct values, so high-cardinality continuous columns get inflated and a coarser but genuinely predictive feature like `fico` is understated. Treat the chart as a hint, not the final word: permutation importance or SHAP values would give a fairer ranking.
</details>

## The End!